In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter

# ==========================================================
# PREPARACIÓN DE LOS DATOS
# ==========================================================

# Normalizar género
df_municipio_genero["genero_limpio"] = (
    df_municipio_genero["genero"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Orden de los tipos de delito
orden_delitos = [
    "Amenazas",
    "Delitos sexuales",
    "Homicidio",
    "Hurto a residencias y entidades comerciales",
    "Hurto de motocicletas y automotores",
    "Lesiones personales",
    "Violencia intrafamiliar"
]

# ----------------------------------------------------------
# Total de registros por municipio y tipo de delito
# Incluye FEMENINO, MASCULINO y NO REPORTA
# ----------------------------------------------------------

totales_municipio_delito = (
    df_municipio_genero
    .groupby(
        ["municipio", "departamento", "tipo_delito"],
        as_index=False
    )["cantidad_delitos"]
    .sum()
    .rename(columns={"cantidad_delitos": "total_delito"})
)

# ----------------------------------------------------------
# Orden de municipios según su volumen total
# ----------------------------------------------------------

orden_municipios = (
    df_municipio_genero
    .groupby(["municipio", "departamento"])["cantidad_delitos"]
    .sum()
    .sort_values(ascending=False)
    .index
)

# ==========================================================
# FUNCIÓN PARA GENERAR EL HEATMAP
# ==========================================================

def crear_heatmap_genero_porcentaje(df, genero, titulo):

    # ------------------------------------------------------
    # Filtrar el género
    # ------------------------------------------------------

    df_genero = df[
        df["genero_limpio"] == genero
    ].copy()

    if df_genero.empty:
        print(f"No se encontraron registros para {genero}.")
        return

    # ------------------------------------------------------
    # Unir el total municipio × delito
    # ------------------------------------------------------

    df_genero = df_genero.merge(
        totales_municipio_delito,
        on=["municipio", "departamento", "tipo_delito"],
        how="left"
    )

    # ------------------------------------------------------
    # Calcular porcentaje
    # ------------------------------------------------------

    df_genero["porcentaje"] = (
        df_genero["cantidad_delitos"]
        / df_genero["total_delito"]
    ) * 100

    # ------------------------------------------------------
    # Matriz de cantidades
    # ------------------------------------------------------

    tabla_cantidad = df_genero.pivot_table(
        index=["municipio", "departamento"],
        columns="tipo_delito",
        values="cantidad_delitos",
        aggfunc="sum",
        fill_value=0
    )

    # ------------------------------------------------------
    # Matriz de porcentajes
    # ------------------------------------------------------

    tabla_porcentaje = df_genero.pivot_table(
        index=["municipio", "departamento"],
        columns="tipo_delito",
        values="porcentaje",
        aggfunc="sum",
        fill_value=0
    )

    # Asegurar orden de columnas
    tabla_cantidad = tabla_cantidad.reindex(
        columns=orden_delitos,
        fill_value=0
    )

    tabla_porcentaje = tabla_porcentaje.reindex(
        columns=orden_delitos,
        fill_value=0
    )

    # Asegurar orden de municipios
    tabla_cantidad = tabla_cantidad.reindex(
        orden_municipios,
        fill_value=0
    )

    tabla_porcentaje = tabla_porcentaje.reindex(
        orden_municipios,
        fill_value=0
    )

    datos = tabla_cantidad.to_numpy()
    porcentajes = tabla_porcentaje.to_numpy()

    # ------------------------------------------------------
    # Escala logarítmica para el color
    # ------------------------------------------------------

    valores_positivos = datos[datos > 0]

    if valores_positivos.size == 0:
        print(f"No hay valores positivos para {genero}.")
        return

    norm = LogNorm(
        vmin=valores_positivos.min(),
        vmax=valores_positivos.max()
    )

    # ======================================================
    # FIGURA
    # ======================================================

    fig, ax = plt.subplots(figsize=(15, 8))

    imagen = ax.imshow(
        datos,
        aspect="auto",
        norm=norm
    )

    # ------------------------------------------------------
    # Eje X
    # ------------------------------------------------------

    etiquetas_delitos = [
        "Amenazas",
        "Delitos\nsexuales",
        "Homicidio",
        "Hurto a residencias\ny entidades \ncomerciales",
        "Hurto de \nmotocicletas y \nautomotores",
        "Lesiones\npersonales",
        "Violencia\nintrafamiliar"
    ]

    ax.set_xticks(np.arange(len(orden_delitos)))
    ax.set_xticklabels(
        etiquetas_delitos,
        rotation=0,
        ha="center"
    )

    ax.set_xlabel(
        "Tipo de delito",
        labelpad=20
    )

    # ------------------------------------------------------
    # Eje Y
    # ------------------------------------------------------

    etiquetas_municipios = [
        f"{municipio} — {departamento}"
        for municipio, departamento in orden_municipios
    ]

    ax.set_yticks(np.arange(len(etiquetas_municipios)))
    ax.set_yticklabels(etiquetas_municipios)

    ax.set_ylabel("Municipio")

    ax.set_title(titulo)

    # ------------------------------------------------------
    # Etiquetas: cantidad + porcentaje
    # ------------------------------------------------------

    for i in range(datos.shape[0]):
        for j in range(datos.shape[1]):

            cantidad = datos[i, j]
            porcentaje = porcentajes[i, j]

            if cantidad > 0:

                texto_cantidad = (
                    f"{int(cantidad):,}".replace(",", ".")
                )

                texto = (
                    f"{texto_cantidad}\n"
                    f"({porcentaje:.1f} %)"
                )
                # Obtener el color de fondo de la celda
                rgba = imagen.cmap(norm(cantidad))

                # Calcular luminancia del color
                r, g, b, _ = rgba

                luminancia = (
                    0.299 * r +
                    0.587 * g +
                    0.114 * b
                )

                # Fondo oscuro -> texto blanco
                # Fondo claro -> texto negro
                color_texto = "white" if luminancia < 0.5 else "black"

                ax.text(
                    j,
                    i,
                    texto,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=color_texto,
                    linespacing=1.1
                )

    # ------------------------------------------------------
    # Barra de color
    # ------------------------------------------------------

    cbar = fig.colorbar(
        imagen,
        ax=ax
    )

    cbar.set_label(
        "Cantidad de registros"
    )

    cbar.ax.yaxis.set_major_formatter(
        FuncFormatter(
            lambda x, pos:
            f"{int(x):,}".replace(",", ".")
        )
    )

    plt.tight_layout()
    plt.show()

In [ ]:
crear_heatmap_genero_porcentaje(
    df_municipio_genero,
    "FEMENINO",
    "Distribución de los registros asociados al género femenino por municipio y tipo de delito, 2014–2024"
)

crear_heatmap_genero_porcentaje(
    df_municipio_genero,
    "MASCULINO",
    "Distribución de los registros asociados al género masculino por municipio y tipo de delito, 2014–2024"
)